# Combining Instrument Properties with Holding Properties using a Derived Property 

In this notebook we will show how you can combine property data from the instrument master with holding properties.
We will first add instruments and their properties to our instrument master, intentionally leaving some property values blank. Then we will be creating a portfolio with sub holding keys, where the sub holding keys will serve as holding properties. Finally, we will create a new derived property that will fill in the gaps of our instrument master's collection of properties with the sub holding key properties.

For more background on sub-holding keys and derived properties you can refer to the below knowledge base articles:

Sub-holding keys: https://support.lusid.com/knowledgebase/article/KA-01879/en-us

Derived properties: https://support.lusid.com/knowledgebase/article/KA-02192/en-us


In [ ]:
# Import generic non-LUSID packages
import os
import pandas as pd
import numpy as np
from datetime import datetime
import json
import pytz
import time
from IPython.core.display import HTML

# Import key modules from the LUSID package
import finbourne.sdk.services.lusid as lu
import finbourne.sdk.services.lusid.models as lm
from finbourne.sdk.extensions import SyncApiClientFactory, RefreshingToken
from finbourne.sdk.exceptions import ApiException

# Import key functions from Lusid-Python-Tools and other packages
from finbourne_sdk_utils.pandas_utils.lusid_pandas import lusid_response_to_data_frame
from finbourne_sdk_utils.cocoon.cocoon import load_from_data_frame
from finbourne_sdk_utils.lpt.lpt import to_date
from finbourne_sdk_utils.cocoon.cocoon_printer import (
    format_instruments_response,
    format_portfolios_response,
    format_transactions_response,
    format_quotes_response,
    format_holdings_response
)

# Set DataFrame display formats
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.options.display.float_format = "{:,.2f}".format
# display(HTML("<style>.container { width:90% !important; }</style>"))

# Set the secrets path
secrets_path = os.getenv("FBN_SECRETS_PATH")

# For running the notebook locally
if secrets_path is None:
    secrets_path = os.path.join(os.path.dirname(os.getcwd()), "secrets.json")

# Authenticate our user and create our API client
api_factory = SyncApiClientFactory(
    access_token=RefreshingToken(), secrets_path=secrets_path
)

print("LUSID Environment Initialised")
print(
    "LUSID API Version :",
    api_factory.build(lu.ApplicationMetadataApi).get_lusid_versions().build_version,
)

In [ ]:
# LUSID Variable Definitions
portfolio_api = api_factory.build(lu.PortfoliosApi)
transaction_portfolios_api = api_factory.build(lu.TransactionPortfoliosApi)
instruments_api = api_factory.build(lu.InstrumentsApi)

In [ ]:
scope = "ibor"

## Adding Instruments with their Properties to the Instrument Master

In [ ]:
instrument_master = pd.read_csv('data/coalesce-demo-instrument-master.csv')
instrument_master

In [ ]:
instrument_identifier_mapping = {
    "ClientInternal": "Ticker",
    "Isin": "Isin",
}

instrument_mapping_required = {"name": "Name"}

responses = load_from_data_frame(
    api_factory=api_factory,
    scope=scope,
    data_frame=instrument_master,
    mapping_required=instrument_mapping_required,
    mapping_optional={},
    file_type="instrument",
    identifier_mapping=instrument_identifier_mapping,
    property_columns=[
        "Strategy",
        "Sector",
    ],
)

succ, failed, errors = format_instruments_response(responses)
pd.DataFrame(
    data=[{"success": len(succ), "failed": len(failed), "errors": len(errors)}]
)

## Creating the Sub Holding Key Properties

In [ ]:
domain = "Transaction"
scope = scope
prop_code = "Strategy"

try:
    api_factory.build(lu.PropertyDefinitionsApi).create_property_definition(
        create_property_definition_request=lm.CreatePropertyDefinitionRequest(
            domain=domain,
            scope=scope,
            code=prop_code,
            value_required=None,
            display_name="Investment strategy",
            data_type_id=lm.ResourceId(scope="system", code="string"),
            life_time=None,
        )
    )

except ApiException as e:
    print(json.loads(e.body)["title"])

In [ ]:
domain = "Transaction"
scope = scope
prop_code = "Sector"

try:
    api_factory.build(lu.PropertyDefinitionsApi).create_property_definition(
        create_property_definition_request=lm.CreatePropertyDefinitionRequest(
            domain=domain,
            scope=scope,
            code=prop_code,
            value_required=None,
            display_name="Sector",
            data_type_id=lm.ResourceId(scope="system", code="string"),
            life_time=None,
        )
    )

except ApiException as e:
    print(json.loads(e.body)["title"])

## Creating the Portfolio

In [ ]:
portfolio_df = pd.read_csv('data/coalesce-demo-portfolio.csv')
portfolio_df

In [ ]:
portfolio_mapping = {
    "required": {"code": "Code", "display_name": "Name", "base_currency": "Currency",},
    "optional": {"created": "Created"},
}

result = load_from_data_frame(
    api_factory=api_factory,
    scope=scope,
    data_frame=portfolio_df,
    mapping_required=portfolio_mapping["required"],
    mapping_optional=portfolio_mapping["optional"],
    file_type="portfolios",
    sub_holding_keys=[
        "Transaction/ibor/Strategy",
        "Transaction/ibor/Sector",
    ],
)

succ, failed = format_portfolios_response(result)
display(pd.DataFrame(data=[{"success": len(succ), "failed": len(failed)}]))

## Adding Transactions to the Portfolio

In [ ]:
transactions = pd.read_csv("data/coalesce-demo-transactions.csv")
transactions

In [ ]:
transaction_field_mapping_required = {
    "code": "portfolio",
    "transaction_id": "txn_id",
    "type": "type",
    "transaction_date": "trade_date",
    "settlement_date": "settlement_date",
    "units": "quantity",
    "transaction_price.price": "price",
    "transaction_price.type": "$Price",
    "total_consideration.amount": "total_consideration",
    "total_consideration.currency": "currency",
    "exchange_rate": "$1",
    "transaction_currency": "currency",
}


transaction_identifier_mapping = {
    "ClientInternal": "Ticker",
}

In [ ]:
responses = load_from_data_frame(
    api_factory=api_factory,
    scope=scope,
    data_frame=transactions,
    mapping_required=transaction_field_mapping_required,
    mapping_optional={},
    identifier_mapping=transaction_identifier_mapping,
    file_type="transaction",
    property_columns = [
        "Strategy", 
        "Sector"
    ],
)

succ, failed = format_transactions_response(responses)
display(pd.DataFrame(data=[{"success": len(succ), "failed": len(failed)}]))

## Creating Derived Properties

In [ ]:
try:
    api_factory.build(
        lu.PropertyDefinitionsApi
    ).create_derived_property_definition(
        create_derived_property_definition_request=lm.CreateDerivedPropertyDefinitionRequest(
            domain="Holding",
            scope=scope,
            code="DerivedStrategy",
            display_name="Strategy from SHK or Instrument properties",
            data_type_id=lm.ResourceId(scope="system", code="string"),
            derivation_formula=f"Coalesce(Properties[Instrument/{scope}/Strategy], SubHoldingKeys[Transaction/{scope}/Strategy], 'None')",
            is_filterable=True,
        )
    )
except ApiException as e:
    print(json.loads(e.body)["title"])

In [ ]:
try:
    api_factory.build(
        lu.PropertyDefinitionsApi
    ).create_derived_property_definition(
        create_derived_property_definition_request=lm.CreateDerivedPropertyDefinitionRequest(
            domain="Holding",
            scope=scope,
            code="DerivedSector",
            display_name="Sector from SHK or Instrument properties",
            data_type_id=lm.ResourceId(scope="system", code="string"),
            derivation_formula=f"if(HoldingType eq 'C') then 'Cash Commitment' else Coalesce(Properties[Instrument/{scope}/Sector], SubHoldingKeys[Transaction/{scope}/Sector], 'None')",
            is_filterable=True,
        )
    )
except ApiException as e:
    print(json.loads(e.body)["title"])

## Retrieve the Portfolio Data

In [ ]:
portfolio = transaction_portfolios_api.get_holdings(
        scope=scope,
        code="coalescePortfolio",
        property_keys=[
            f"Instrument/{scope}/Strategy", 
            f"Instrument/{scope}/Sector", 
            f"Holding/{scope}/DerivedStrategy", 
            f"Holding/{scope}/DerivedSector",
            "Instrument/default/ClientInternal",
            ]
    )

In [ ]:
results = lusid_response_to_data_frame(portfolio)

# We filtered out the relevant columns to make our use case more clear and readable.
results[[
    'instrument_uid',
    'properties.Instrument/default/ClientInternal.value.label_value',
    'properties.Instrument/ibor/Sector.value.label_value',
    'sub_holding_keys.Transaction/ibor/Sector.value.label_value',
    'properties.Holding/ibor/DerivedSector.value.label_value',
    'properties.Instrument/ibor/Strategy.value.label_value',
    'sub_holding_keys.Transaction/ibor/Strategy.value.label_value',
    'properties.Holding/ibor/DerivedStrategy.value.label_value',   
    ]]

Here we can see the Instrument properties, which header starts with "properties.Instrument" and the Sub Holding Keys, which header starts with "sub_holding_keys.Transaction".
When we look at MSFT, we can see the NaN value for Sector in the instrument master ('properties.Instrument/ibor/Sector.value.label_value').
However, there is a value for Sector in the Sub Holding Keys ('sub_holding_keys.Transaction/ibor/Sector.value.label_value') and therefore we see the value "Technology" in the derived property for sector ('properties.Holding/ibor/DerivedSector.value.label_value').